[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/semiconductors.ipynb)

# Semiconductor electronics full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **solvers and simulations**: a p-n junction solved from two laws on a grid, with
its carriers, its bare ions, its field and its barrier; the diode's curve from the diode equation, on plain and on
powers-of-ten axes; a diode and a resistor solved together; the three rectifiers, computed from the AC wave; a
smoothing capacitor stepped through time, with its ripple; and the phone charger's first stage. You can move all of
them yourself: the **doping**, the diode's **I₀**, the **battery** and the **resistor**, the **supply voltage**, the
**capacitor** and the **load**, and the **mains** the charger is plugged into.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `5e22` for 5 × 10²²,
`1e-14` for 1 × 10⁻¹⁴ (`5*10^22` works too). You can type the arithmetic: `1e32/2e22`, `(6-0.7)/530`,
`0.08/(2*50*2200e-6)`, `15*sqrt(2)`, `0.02585*log(2)`. Numbers without thousands separators: `2500`, not `2,500`. For
several answers, separate them with commas, in the order the question lists them (the box's label shows the order).
You can also put your answer between the quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it
checks straight away.

**Safety.** Mains voltage can kill, and a charger's capacitors can stay charged after it is unplugged. Everything
here is arithmetic and simulation: never open a charger, and never build a mains circuit as an experiment.

**The cheat sheet.** Every formula and constant from the course on two pages:
[semiconductor electronics cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/semiconductors_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the formulas, the solvers and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # curves in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# an electron MINT · a hole CORAL · current, and only current, SKY · the one mark GOLD (as in the course)

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="solved in code · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
E_CH = 1.60e-19            # C, the size of an electron's charge
VT = 0.02585               # V, the thermal voltage kT/e at 300 K; kT = 0.02585 eV ("a fortieth of an electron volt")
EPS0 = 8.85e-12            # F/m
EPS_SI = 11.7              # silicon's relative permittivity
NI = 1e16                  # per m³, free electrons (= holes) in pure silicon at 300 K
ATOMS = 5e28               # per m³, atoms in silicon
GAP = {"Si": 1.12, "Ge": 0.66, "diamond": 5.5}        # eV
I0 = 1e-14                 # A, the course's diode: its reverse saturation current
DROP = 0.7                 # V, a conducting silicon diode

def close(a, b, rel=1e-4):
    """True when a solved or simulated value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def minority(n_major, ni=NI): return ni * ni / n_major                             # n_e n_h = n_i²
def shockley(V, i0=I0, vt=VT): return i0 * (math.exp(min(V / vt, 700.0)) - 1.0)    # I = I₀ (e^(V/V_T) − 1)
def diode_v(I, i0=I0, vt=VT): return vt * math.log(I / i0 + 1.0)                   # the same, turned round
def model_current(V, R, drop=DROP, n=1): return max(0.0, (V - n * drop) / R)       # I = (V − 0.7)/R
def peak(rms): return rms * math.sqrt(2)                                           # peak = r m s × √2
def ripple(I, f, C, pulses=2): return I / (pulses * f * C)                         # ΔV = I/(2fC); half-wave: I/(fC)
def barrier(Na, Nd, ni=NI, vt=VT): return vt * math.log(Na * Nd / (ni * ni))       # V₀ = V_T ln(N_A N_D / n_i²)
def depletion_width(V0, Na, Nd): return math.sqrt(2 * EPS_SI * EPS0 * V0 * (1 / Na + 1 / Nd) / E_CH)

def solve_loop(V, R, n=1, i0=I0):
    """A battery V, a resistor R and n forward diodes in one loop: the current that satisfies the diode equation and
    the loop rule together, V = n V_T ln(I/I₀ + 1) + I R (found by halving: the right side only grows with I)."""
    if V <= 0: return 0.0
    lo, hi = 0.0, V / R
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if n * diode_v(mid, i0) + mid * R > V: hi = mid
        else: lo = mid
    return 0.5 * (lo + hi)

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "log": math.log, "ln": math.log}

def calc(text):
    """Plain arithmetic only: numbers (5e22 too), + − × ÷, powers, brackets, sqrt, exp, log (natural) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.Name) and n.id == "e": return math.e
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(MV/m|V/m|per m³|per m3|m⁻³|µF|uF|mF|F|mV|kV|V|mA|µA|uA|A|Ω|ohms?|kΩ|eV|K|Hz|µm|um|mm|m|W|%|times|"
        r"millions?|pulses per second|a second)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Bands and pure semiconductors',
     "q": "Germanium's band gap is 0.66 eV. At room temperature the thermal energy is 0.02585 eV. How many times bigger is the gap?",
     "parts": ['ratio'],
     "answer": [25.53191489361702], "tol": [0.005], "function": '—',
     "hint": 'divide the gap by the thermal energy',
     "working": '0.66 / 0.02585 = 25.53 times'},
    {"id": 'P1.2', "tier": 1, "group": 'Bands and pure semiconductors',
     "q": 'Pure germanium holds 4.4 × 10²⁸ atoms and 2 × 10¹⁹ free electrons in each cubic metre. How many atoms are there for each free electron, in units of 10⁹?',
     "parts": ['atoms per free electron (× 10⁹)'],
     "answer": [2.2], "tol": [0.005], "function": '—',
     "hint": 'divide the atoms by the free electrons',
     "working": '4.4 × 10²⁸ / 2 × 10¹⁹ = 2.2 × 10⁹ atoms for each free electron'},
    {"id": 'P1.3', "tier": 1, "group": 'Bands and pure semiconductors',
     "q": 'Pure silicon holds 1 × 10¹⁶ free electrons in each cubic metre. Counting electrons and holes together, how many charge carriers is that, in units of 10¹⁶ per m³?',
     "parts": ['carriers (× 10¹⁶ per m³)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'in a pure semiconductor, holes and free electrons come in pairs',
     "working": 'every freed electron leaves one hole: 1 × 10¹⁶ + 1 × 10¹⁶ = 2 × 10¹⁶ per m³'},
    {"id": 'P1.4', "tier": 1, "group": 'Doping',
     "q": 'An n-type silicon crystal has 2 × 10²² free electrons per m³. Pure silicon has n_i = 1 × 10¹⁶ per m³. How many holes per m³ does the crystal hold, in units of 10⁹?',
     "parts": ['holes (× 10⁹ per m³)'],
     "answer": [5.0], "tol": [0.005], "function": '—',
     "hint": 'n_e × n_h = n_i²',
     "working": 'n_h = n_i² / n_e = 10³² / 2 × 10²² = 5 × 10⁹ per m³'},
    {"id": 'P1.5', "tier": 1, "group": 'Doping',
     "q": 'Silicon has 5 × 10²⁸ atoms per m³. One atom in every 2 million is swapped for phosphorus. How many phosphorus atoms per m³ is that, in units of 10²²?',
     "parts": ['donors (× 10²² per m³)'],
     "answer": [2.5], "tol": [0.005], "function": '—',
     "hint": 'divide the atoms by two million',
     "working": '5 × 10²⁸ / 2 × 10⁶ = 2.5 × 10²² per m³'},
    {"id": 'P1.6', "tier": 1, "group": 'Junction and diode',
     "q": "A junction's barrier is 0.6 V, across a depletion region 0.3 µm wide. What is the average electric field there, in millions of volts per metre?",
     "parts": ['field (MV/m)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'field = voltage ÷ distance, with the distance in metres',
     "working": 'E = V/d = 0.6 / 0.3 × 10⁻⁶ = 2 × 10⁶ V/m'},
    {"id": 'P1.7', "tier": 1, "group": 'Junction and diode',
     "q": 'A 6 V battery, a 530 Ω resistor and a silicon diode (forward, dropping 0.7 V) form one loop. What is the current, in mA?',
     "parts": ['current (mA)'],
     "answer": [10.0], "tol": [0.005], "function": '—',
     "hint": 'the resistor gets what the diode leaves: 6 − 0.7 volts',
     "working": 'I = (6 − 0.7) / 530 = 0.01 A = 10 mA'},
    {"id": 'P1.8', "tier": 1, "group": 'Junction and diode',
     "q": "A diode carries 2 mA at 0.68 V. Use the diode equation's rule, I grows by e^(ΔV/V_T) with V_T = 25.85 mV, to find its current at 0.74 V, in mA.",
     "parts": ['current (mA)'],
     "answer": [20.373404569159977], "tol": [0.005], "function": 'exp',
     "hint": 'sixty millivolts more multiplies the current by about ten',
     "working": 'ΔV = 0.06 V; e^(0.06/0.02585) = 10.187; 2 × 10.187 = 20.37 mA (sixty millivolts more: about ten times)'},
    {"id": 'P1.9', "tier": 1, "group": 'Rectifiers',
     "q": 'A transformer gives 15 V rms. What is the peak of its voltage wave, in volts?',
     "parts": ['peak (V)'],
     "answer": [21.213203435596427], "tol": [0.005], "function": 'sqrt',
     "hint": 'peak = rms × √2',
     "working": 'peak = rms × √2 = 15 × 1.4142 = 21.21 V'},
    {"id": 'P1.10', "tier": 1, "group": 'Rectifiers',
     "q": 'A full-wave rectifier runs from a 60 Hz supply. How many pulses does its output have each second?',
     "parts": ['pulses per second'],
     "answer": [120.0], "tol": [0.005], "function": '—',
     "hint": 'a full-wave rectifier uses both half-cycles',
     "working": 'both halves of every cycle become a pulse: 2 × 60 = 120 a second'},
    {"id": 'P2.1', "tier": 2, "group": 'Bands and pure semiconductors',
     "q": 'At 300 K the thermal energy is 0.02585 eV, and it grows in step with the temperature in kelvin. At what temperature is it 0.05 eV, in kelvin?',
     "parts": ['temperature (K)'],
     "answer": [580.2707930367505], "tol": [0.005], "function": '—',
     "hint": 'kT is proportional to T',
     "working": 'T = 300 × 0.05 / 0.02585 = 580.27 K'},
    {"id": 'P2.2', "tier": 2, "group": 'Bands and pure semiconductors',
     "q": 'In pure silicon, equal numbers of electrons and holes carry a total current of 2.4 mA. The electrons drift three times as fast as the holes. How much current does each carry, in mA (electrons first)?',
     "parts": ['electron current (mA)', 'hole current (mA)'],
     "answer": [1.8, 0.6], "tol": [0.005, 0.005], "function": '—',
     "hint": 'split the total in the ratio 3 : 1',
     "working": 'equal numbers, so the currents are in the ratio of the speeds, 3 to 1: electrons 3/4 × 2.4 = 1.8 mA, holes 1/4 × 2.4 = 0.6 mA; both run the same way round the circuit and add'},
    {"id": 'P2.3', "tier": 2, "group": 'Doping',
     "q": 'Silicon (5 × 10²⁸ atoms per m³, n_i = 1 × 10¹⁶ per m³) is doped with one boron atom per 5 million silicon atoms. Find the holes per m³ (in units of 10²²) and the free electrons per m³ (in units of 10¹⁰).',
     "parts": ['holes (× 10²² per m³)', 'electrons (× 10¹⁰ per m³)'],
     "answer": [1.0, 1.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'each acceptor gives one hole; then n_e × n_h = n_i²',
     "working": 'N_A = 5 × 10²⁸ / 5 × 10⁶ = 1 × 10²² per m³, and each boron atom gives one hole: n_h = 1 × 10²²; n_e = n_i² / n_h = 10³² / 10²² = 1 × 10¹⁰ per m³'},
    {"id": 'P2.4', "tier": 2, "group": 'Doping',
     "q": 'Pure silicon has 2 × 10¹⁶ carriers per m³ (electrons and holes together). Doped with 1 × 10²³ donors per m³, how many times as many carriers does it have? Answer in millions.',
     "parts": ['times as many (millions)'],
     "answer": [5.0], "tol": [0.005], "function": '—',
     "hint": "nearly all the carriers are now the donors' electrons",
     "working": '1 × 10²³ / 2 × 10¹⁶ = 5 × 10⁶: five million times as many (the few holes left hardly count)'},
    {"id": 'P2.5', "tier": 2, "group": 'Junction and diode',
     "q": 'A diode has I₀ = 1 × 10⁻¹⁴ A, and V_T = 25.85 mV. What current does the diode equation give at 0.65 V, in mA?',
     "parts": ['current (mA)'],
     "answer": [0.8324614831475218], "tol": [0.005], "function": 'exp',
     "hint": 'V/V_T = 0.65 / 0.02585; the −1 hardly matters here',
     "working": 'I = I₀ (e^(V/V_T) − 1) = 10⁻¹⁴ × (e^(25.145) − 1) = 0.83 mA'},
    {"id": 'P2.6', "tier": 2, "group": 'Junction and diode',
     "q": 'The same diode carries 27.07 mA at 0.74 V and 58.67 mA at 0.76 V. What is its dynamic resistance between those two points, in ohms?',
     "parts": ['dynamic resistance (Ω)'],
     "answer": [0.6329113924050632], "tol": [0.005], "function": '—',
     "hint": 'a small change in voltage over the change in current it causes',
     "working": 'r = ΔV/ΔI = 0.02 V / (0.05867 − 0.02707) A = 0.63 Ω'},
    {"id": 'P2.7', "tier": 2, "group": 'Junction and diode',
     "q": 'A 12 V battery must send 25 mA through a silicon diode (0.7 V when it conducts). What resistor goes in the loop, in ohms?',
     "parts": ['resistance (Ω)'],
     "answer": [452.0], "tol": [0.005], "function": '—',
     "hint": 'the resistor must drop 12 − 0.7 volts at 25 mA',
     "working": 'R = (12 − 0.7) / 0.025 = 452 Ω'},
    {"id": 'P2.8', "tier": 2, "group": 'Rectifiers',
     "q": 'A half-wave rectifier (one silicon diode, 0.7 V) runs from a 6 V rms, 50 Hz transformer. What is the peak of its output (V), and how many pulses does it give each second?',
     "parts": ['output peak (V)', 'pulses per second'],
     "answer": [7.785281374238571, 50.0], "tol": [0.005, 0.005], "function": 'sqrt',
     "hint": 'one diode: one drop, and only one half of each cycle',
     "working": 'peak in = 6 × √2 = 8.49 V; out = 8.49 − 0.7 = 7.79 V; one pulse for every cycle: 50 a second'},
    {"id": 'P2.9', "tier": 2, "group": 'Rectifiers',
     "q": 'A bridge rectifier on a 50 Hz supply feeds 80 mA to its load, with a 2,200 µF smoothing capacitor. What is the ripple, in volts?',
     "parts": ['ripple (V)'],
     "answer": [0.36363636363636365], "tol": [0.005], "function": '—',
     "hint": 'ΔV = I ÷ (2 f C), with C in farads',
     "working": 'ΔV = I/(2fC) = 0.080 / (2 × 50 × 0.0022) = 0.36 V'},
    {"id": 'P2.10', "tier": 2, "group": 'The phone charger',
     "q": 'In a country with 240 V rms mains, what is the peak of the mains wave (V), and what peak reaches the capacitor after a bridge of silicon diodes (0.7 V each)?',
     "parts": ['mains peak (V)', 'peak at the capacitor (V)'],
     "answer": [339.4112549695428, 338.01125496954285], "tol": [0.005, 0.005], "function": 'sqrt',
     "hint": 'in a bridge the current always passes two diodes',
     "working": 'peak = 240 × √2 = 339.41 V; two diodes are in the path: 339.41 − 1.4 = 338.01 V'},
    {"id": 'P3.1', "tier": 3, "group": 'Bands and pure semiconductors',
     "q": 'The number of free electrons in pure silicon grows as T^1.5 × e^(−E_g/(2kT)), with E_g = 1.12 eV and kT = 0.02585 eV at 300 K. How many times as many are there at 310 K as at 300 K?',
     "parts": ['ratio'],
     "answer": [2.1127818135922753], "tol": [0.005], "function": 'exp',
     "hint": 'work out kT at 310 K first: it grows in step with T',
     "working": '(310/300)^1.5 = 1.0504; kT at 310 K = 0.02671 eV; the exponents differ by 1.12/2 × (1/0.02585 − 1/0.02671) = 0.6988; e to that = 2.0114; together 2.11 times: ten degrees more than doubles them'},
    {"id": 'P3.2', "tier": 3, "group": 'Doping',
     "q": "A silicon crystal (n_i = 1 × 10¹⁶ per m³) holds 7 × 10²² donors and 2 × 10²² acceptors per m³. Each acceptor takes one donor's electron. Find the free electrons per m³ (in units of 10²²) and the holes per m³ (in units of 10⁹).",
     "parts": ['electrons (× 10²² per m³)', 'holes (× 10⁹ per m³)'],
     "answer": [5.0, 2.0000000000000004], "tol": [0.005, 0.005], "function": '—',
     "hint": 'the acceptors soak up electrons first; what is left decides the type',
     "working": 'electrons left free: 7 × 10²² − 2 × 10²² = 5 × 10²² per m³ (it is n-type); n_h = 10³² / 5 × 10²² = 2 × 10⁹ per m³'},
    {"id": 'P3.3', "tier": 3, "group": 'Doping',
     "q": 'A p-type silicon crystal (5 × 10²⁸ atoms per m³, n_i = 1 × 10¹⁶ per m³) holds only 2.5 × 10⁹ free electrons per m³. How many acceptors per m³ does it hold (in units of 10²²), and that is one acceptor per how many million silicon atoms?',
     "parts": ['acceptors (× 10²² per m³)', 'silicon atoms per acceptor (millions)'],
     "answer": [4.0, 1.25], "tol": [0.005, 0.005], "function": '—',
     "hint": 'run n_e × n_h = n_i² backwards',
     "working": 'n_h = n_i² / n_e = 10³² / 2.5 × 10⁹ = 4 × 10²² per m³, one hole from each acceptor; 5 × 10²⁸ / 4 × 10²² = 1.25 × 10⁶: one acceptor per 1.25 million atoms'},
    {"id": 'P3.4', "tier": 3, "group": 'Junction and diode',
     "q": "A formula from beyond the course gives a junction's barrier: V₀ = V_T × ln(N_A × N_D / n_i²). For silicon with N_A = 1 × 10²³, N_D = 1 × 10²¹ and n_i = 1 × 10¹⁶ per m³, and V_T = 0.02585 V, what is V₀, in volts?",
     "parts": ['barrier (V)'],
     "answer": [0.714261895846753], "tol": [0.005], "function": 'log',
     "hint": 'ln is the natural logarithm: math.log',
     "working": 'N_A N_D / n_i² = 10⁴⁴ / 10³² = 10¹²; ln(10¹²) = 27.631; V₀ = 0.02585 × 27.631 = 0.71 V'},
    {"id": 'P3.5', "tier": 3, "group": 'Junction and diode',
     "q": "By how many millivolts must a diode's forward voltage rise to double its current? (V_T = 25.85 mV)",
     "parts": ['voltage step (mV)'],
     "answer": [17.917854617474585], "tol": [0.005], "function": 'log',
     "hint": 'set e^(ΔV/V_T) equal to 2',
     "working": 'e^(ΔV/V_T) = 2, so ΔV = V_T × ln 2 = 25.85 × 0.6931 = 17.92 mV'},
    {"id": 'P3.6', "tier": 3, "group": 'Junction and diode',
     "q": 'A 3 V battery, a 1 kΩ resistor and our diode (I₀ = 1 × 10⁻¹⁴ A, V_T = 0.02585 V) form a loop. The 0.7 V model gives 2.3 mA. What voltage does the diode equation say the diode needs at 2.3 mA (V)? With that voltage in place of 0.7, what is the current (mA)?',
     "parts": ['diode voltage (V)', 'current (mA)'],
     "answer": [0.6762707720208417, 2.3237292279791584], "tol": [0.005, 0.005], "function": 'log',
     "hint": 'turn the diode equation round: V = V_T ln(I/I₀)',
     "working": 'V = V_T ln(I/I₀) = 0.02585 × ln(2.3 × 10¹¹) = 0.6763 V; I = (3 − 0.6763) / 1000 = 2.324 mA: the model was within about one percent'},
    {"id": 'P3.7', "tier": 3, "group": 'Rectifiers',
     "q": 'A load draws 150 mA from a 60 Hz supply, and the ripple must stay at 0.5 V. What capacitor does a bridge rectifier need, in µF? And a half-wave rectifier?',
     "parts": ['bridge (µF)', 'half-wave (µF)'],
     "answer": [2500.0, 5000.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'a half-wave rectifier refills the capacitor half as often',
     "working": 'bridge: C = I/(2 f ΔV) = 0.15 / (2 × 60 × 0.5) = 0.0025 F = 2,500 µF; half-wave: the capacitor waits a whole cycle, C = I/(f ΔV) = 5,000 µF'},
    {"id": 'P3.8', "tier": 3, "group": 'The phone charger',
     "q": "The course's charger (14.7 µF after its bridge, drawing 6.9 W) is plugged into 100 V rms, 50 Hz mains. Find the peak voltage at the capacitor (V, two 0.7 V drops), the current the next stage draws at that voltage (mA), and the ripple (V).",
     "parts": ['peak (V)', 'current (mA)', 'ripple (V)'],
     "answer": [140.0213562373095, 49.27819716519397, 33.522583105574135], "tol": [0.005, 0.005, 0.005], "function": 'sqrt',
     "hint": 'lower mains: less voltage, so more current for the same power, and a bigger ripple',
     "working": 'peak = 100 × √2 − 1.4 = 140.02 V; I = P/V = 6.9 / 140.02 = 49.28 mA; ΔV = I/(2fC) = 0.04928 / (2 × 50 × 14.7 × 10⁻⁶) = 33.52 V: nearly a quarter of the peak'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

print("Ready: kT =", VT, "eV at 300 K; the diode's I0 =", I0, "A; 0.70 V gives", round(shockley(0.70) * 1e3, 2), "mA")

## Tier 1 · Warm-up

One idea, one step: a band gap against the thermal energy, how rare a broken bond is, electrons and holes in pairs,
the mass action law, a doping level, a junction's field, a diode in a loop, the sixty-millivolt rule, a wave's peak,
and a full-wave rectifier's pulses.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `minority(n)`, `shockley(V)`, `diode_v(I)`,
`model_current(V, R)`, `peak(rms)`, `ripple(I, f, C)` and the rest. Work in volts, amps, farads, metres and "per cubic
metre" (mA is `1e-3`, µF is `1e-6`, µm is `1e-6`), then change the unit at the end. Each line below computes one
warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.1  gap / kT = {GAP['Ge'] / VT:.2f};  P1.8  2 mA × e^(0.06/V_T) = {2 * math.exp(0.06 / VT):.2f} mA")
assert agrees("P1.1", [GAP["Ge"] / VT]) and agrees("P1.2", [4.4e28 / 2e19 / 1e9]) and agrees("P1.3", [2])
assert agrees("P1.4", [minority(2e22) / 1e9]) and agrees("P1.5", [ATOMS / 2e6 / 1e22])
assert agrees("P1.6", [0.6 / 0.3e-6 / 1e6]) and agrees("P1.7", [model_current(6, 530) * 1e3])
assert agrees("P1.8", [2 * math.exp(0.06 / VT)]) and agrees("P1.9", [peak(15)]) and agrees("P1.10", [2 * 60])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## The p-n junction, solved from two laws

One crystal of silicon: acceptors on the left (p-type), donors on the right (n-type). Nothing here is drawn by hand.
The picture is **solved** on a grid from two laws alone:

- **charge makes field** (Poisson's equation): wherever the fixed ions are not covered by carriers, the voltage curves;
- **carriers avoid high energy** (the Boltzmann factor): electrons thin out as e^(ψ/V_T) falls, holes as e^(−ψ/V_T).

Out come the carriers (a powers-of-ten scale: they thin out by a factor of a million million), the two strips of bare
ions, the field (it points from n to p) and the voltage step: the **barrier**. The solved barrier, width and peak
field are then set beside the formulas. Move the doping and watch the depletion region narrow as it rises.

In [ ]:
def solve_junction(Na, Nd, ni=NI, half=1.5e-6, n=601):
    """p side x < 0, n side x > 0. u = ψ/V_T; u″ = (e/(ε V_T)) (n − p − N), with n = n_i eᵘ, p = n_i e⁻ᵘ (Newton's method)."""
    dx = 2 * half / (n - 1); x = np.linspace(-half, half, n)
    dop = np.where(x < -1e-15, -Na, np.where(x > 1e-15, Nd, 0.5 * (Nd - Na)))
    u = np.arcsinh(dop / (2 * ni))                           # start: every point neutral by itself
    k = E_CH * dx * dx / (EPS_SI * EPS0 * VT)
    for _ in range(400):
        eu, emu = np.exp(u), np.exp(-u)
        f = np.zeros(n); b = np.ones(n); a = np.zeros(n); c = np.zeros(n)
        f[1:-1] = -(u[:-2] - 2 * u[1:-1] + u[2:] - k * (ni * (eu[1:-1] - emu[1:-1]) - dop[1:-1]))
        a[1:-1] = 1.0; c[1:-1] = 1.0; b[1:-1] = -2.0 - k * ni * (eu[1:-1] + emu[1:-1])
        for i in range(1, n):                                # the Thomas algorithm: a tridiagonal solve
            w = a[i] / b[i - 1]; b[i] -= w * c[i - 1]; f[i] -= w * f[i - 1]
        du = np.zeros(n); du[-1] = f[-1] / b[-1]
        for i in range(n - 2, -1, -1): du[i] = (f[i] - c[i] * du[i + 1]) / b[i]
        big = float(np.max(np.abs(du)))
        u = u + min(1.0, 1.0 / big) * du if big > 0 else u   # damped: never more than one V_T a step
        if big < 1e-10: break
    psi = VT * u
    ne, nh = ni * np.exp(u), ni * np.exp(-u)
    rho = E_CH * (nh - ne + dop)
    return {"x": x, "psi": psi, "n": ne, "p": nh, "rho": rho, "E": -np.gradient(psi, x), "dx": dx}

def junction_picture(log_doping=22.0):
    N = 10 ** log_doping
    half = max(0.3e-6, 3.5 * depletion_width(barrier(N, N), N, N))
    J = solve_junction(N, N, half=half)
    x = J["x"] * 1e6
    V0 = J["psi"][-1] - J["psi"][0]
    charge = float(np.sum(J["rho"][J["x"] > 0]) * J["dx"])
    width = 2 * charge / (E_CH * N)
    fig, axes = plt.subplots(3, 1, figsize=(10, 8.4), sharex=True)
    fig.subplots_adjust(left=0.11, right=0.97, top=0.90, bottom=0.08, hspace=0.18)
    axes[0].semilogy(x, J["n"], color=MINT, label="free electrons"); axes[0].semilogy(x, J["p"], color=CORAL, label="holes")
    axes[0].set_ylabel("per m³"); axes[0].legend(frameon=False, loc="center right")
    axes[1].fill_between(x, J["rho"] / (E_CH * N), color=DIM, alpha=0.6); axes[1].set_ylabel("bare ions (÷ doping)")
    axes[1].text(0.02, 0.1, "−  on the p side", transform=axes[1].transAxes, color=CHALK)
    axes[1].text(0.98, 0.85, "+  on the n side", transform=axes[1].transAxes, color=CHALK, ha="right")
    axes[2].plot(x, J["psi"] - J["psi"][0], color=CHALK); axes[2].set_ylabel("voltage step (V)"); axes[2].set_xlabel("position (µm):  p side  |  n side")
    axes[2].annotate("", xy=(-0.05 * half * 1e6, 0.5 * V0), xytext=(0.05 * half * 1e6, 0.5 * V0), arrowprops=dict(arrowstyle="->", color=CHALK))
    axes[2].text(0, 0.62 * V0, "E", color=CHALK, ha="center", family="monospace")
    fig.suptitle(f"doping 10^{log_doping:.1f} per m³ each side · barrier {V0:.3f} V · width {width * 1e6:.3f} µm · "
                 f"peak field {abs(J['E']).max() / 1e6:.2f} MV/m", x=0.03, ha="left", color=GOLD, fontsize=13)
    footnote(fig); plt.show()
    return V0, width, float(J["E"].min())

V0, width, E_min = junction_picture(22.0)
assert close(V0, barrier(1e22, 1e22), 1e-6)                  # the solved barrier is V_T ln(N_A N_D / n_i²): 0.714 V
assert abs(width / depletion_width(V0, 1e22, 1e22) - 1) < 0.12   # the bare ions' width against the formula's 0.43 µm
assert E_min < 0                                             # the field is negative: it points to the left, from n to p
assert agrees("P3.4", [barrier(1e23, 1e21)])                 # the barrier formula, as the stretch question uses it

In [ ]:
interact(junction_picture, log_doping=FloatSlider(value=22.0, min=21.0, max=23.5, step=0.25, description="log₁₀ doping"));

## Tier 2 · Core

The level of the course's worked problems: the temperature for a given thermal energy, electrons' and holes' shares
of a current, p-type doping, how many times more carriers, the diode equation, dynamic resistance, choosing a
resistor, a half-wave rectifier, a bridge's ripple, and the mains' peak.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

The diode equation is one line, `shockley(V)`. The dynamic resistance is a small change in voltage over the change in
current it causes, read off two points of the curve. Each line below computes one core answer and checks it.

In [ ]:
print(f"P2.5  I(0.65 V) = {shockley(0.65) * 1e3:.2f} mA;  P2.6  r_d = {0.02 / (shockley(0.76) - shockley(0.74)):.2f} Ω")
assert agrees("P2.1", [300 * 0.05 / VT]) and agrees("P2.2", [2.4 * 3 / 4, 2.4 / 4])
assert agrees("P2.3", [ATOMS / 5e6 / 1e22, minority(ATOMS / 5e6) / 1e10]) and agrees("P2.4", [1e23 / 2e16 / 1e6])
assert agrees("P2.5", [shockley(0.65) * 1e3]) and agrees("P2.6", [0.02 / (shockley(0.76) - shockley(0.74))])
assert agrees("P2.7", [(12 - DROP) / 0.025]) and agrees("P2.8", [peak(6) - DROP, 50])
assert agrees("P2.9", [ripple(0.080, 50, 2200e-6)]) and agrees("P2.10", [peak(240), peak(240) - 2 * DROP])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## The diode's curve, on two kinds of axes

The diode equation, I = I₀ (e^(V/V_T) − 1), plotted twice. On plain axes (left) the current hugs zero and then rears
up near 0.7 V. On a powers-of-ten current axis (right) the same curve is a straight line: every 60 millivolts more
multiplies the current by ten. Move I₀: a diode with a bigger I₀ (like germanium) starts to conduct at a lower voltage.

In [ ]:
def diode_picture(log_i0=-14.0):
    i0 = 10 ** log_i0
    V = np.linspace(-0.3, 0.85, 500)
    I = np.array([shockley(v, i0) for v in V])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.15, wspace=0.26)
    a1.plot(V, np.clip(I, None, 0.3) * 1e3, color=CHALK); a1.set_ylim(-10, 300)
    a1.set_xlabel("voltage across the diode (V)"); a1.set_ylabel("current (mA)")
    v10 = diode_v(0.010, i0)
    a1.plot([v10], [10], "o", color=GOLD); a1.annotate(f"10 mA at {v10:.2f} V", (v10, 10), xytext=(-110, 40), textcoords="offset points", color=GOLD)
    Vf = V[V > 0.05]
    a2.semilogy(Vf, [shockley(v, i0) for v in Vf], color=CHALK)
    a2.set_xlabel("voltage across the diode (V)"); a2.set_ylabel("current (A), powers of ten")
    slope = VT * math.log(10) * 1e3
    fig.suptitle(f"I₀ = 10^{log_i0:.1f} A · reverse current −I₀ · a tenfold step every {slope:.1f} mV", x=0.03, ha="left", color=GOLD)
    footnote(fig, "the diode equation · an ideal diode"); plt.show()
    return v10, slope

v10, slope = diode_picture(-14.0)
assert abs(v10 - 0.714) < 0.001 and abs(slope - 59.5) < 0.05     # 10 mA at 0.714 V; a decade every 59.5 mV
assert close(shockley(0.76) / shockley(0.70), math.exp(0.06 / VT), 1e-9) and agrees("P3.5", [VT * math.log(2) * 1e3])
assert shockley(0.0) == 0.0 and close(shockley(-5.0), -I0, 1e-9)  # the two edges: zero at zero, −I₀ in reverse

In [ ]:
interact(diode_picture, log_i0=FloatSlider(value=-14.0, min=-15.0, max=-6.0, step=0.5, description="log₁₀ I₀ (A)"));

## A diode and a resistor, solved together

A battery, a resistor and the diode in one loop. Two things must both be true: the diode equation (the curve), and the
loop rule, V = I R + V(diode) (the straight line, called the load line). The loop settles where they cross. The
course's shortcut, "the diode keeps 0.7 V", is the dashed mark: see how close it lands, and how it fails for a weak
battery.

In [ ]:
def loop_picture(V=5.0, R=430.0):
    I = solve_loop(V, R); vd = diode_v(I) if I > 0 else V
    model = model_current(V, R)
    vs = np.linspace(0, max(1.0, min(V, 1.2)), 400)
    fig, ax = plt.subplots(figsize=(9.5, 4.4))
    fig.subplots_adjust(left=0.1, right=0.97, top=0.84, bottom=0.15)
    top = max(1.4 * max(I, model), 1e-3)
    ax.plot(vs, np.clip([shockley(v) for v in vs], None, 2 * top) * 1e3, color=CHALK, label="the diode equation")
    ax.plot(vs, (V - vs) / R * 1e3, color=SKY, label="the loop rule: I = (V − V_diode)/R")
    ax.plot([vd], [I * 1e3], "o", color=GOLD)
    ax.axvline(DROP, color=DIM, linestyle="--"); ax.set_ylim(0, top * 1e3)
    ax.set_xlabel("voltage across the diode (V)"); ax.set_ylabel("current (mA)"); ax.legend(frameon=False, loc="upper left")
    off = abs(I / model - 1) * 100 if model > 0 else float("nan")
    fig.suptitle(f"solved: {I * 1e3:.3f} mA, diode {vd:.3f} V · the 0.7 V model: {model * 1e3:.3f} mA"
                 + (f" ({off:.1f} % apart)" if model > 0 else " (below 0.7 V: no current)"), x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return I, model

I_exact, I_model = loop_picture(5.0, 430.0)
assert abs(I_exact * 1e3 - 9.97) < 0.005 and close(I_model, 0.010, 1e-9)       # 9.97 mA solved, 10 mA by the model
_v36 = diode_v(model_current(3, 1000)); assert agrees("P3.6", [_v36, (3 - _v36) / 1000 * 1e3])
assert solve_loop(0.5, 430.0) < 3e-6 and model_current(0.5, 430.0) == 0.0      # a weak battery: next to nothing

In [ ]:
interact(loop_picture, V=FloatSlider(value=5.0, min=0.2, max=12.0, step=0.1, description="battery (V)"),
         R=FloatSlider(value=430.0, min=50.0, max=2000.0, step=10.0, description="resistor (Ω)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, or spot the trap: carriers ten degrees warmer, a crystal with both kinds
of dopant, doping from a minority count, a junction's barrier, the step that doubles a current, a better answer than
the 0.7 V model, capacitors for two kinds of rectifier, and the charger on low mains.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is one line of algebra, then one line of Python. The trap in P3.2: the acceptors soak up
donor electrons first, and only what is left decides the type. The trap in P3.8: lower mains means more current for
the same power, so a bigger ripple.

In [ ]:
kT310 = VT * 310 / 300
ratio = (310 / 300) ** 1.5 * math.exp(GAP["Si"] / 2 * (1 / VT - 1 / kT310))
print(f"P3.1  ten degrees warmer: {ratio:.2f} times as many free electrons")
assert agrees("P3.1", [ratio]) and agrees("P3.2", [(7e22 - 2e22) / 1e22, minority(5e22) / 1e9])
assert agrees("P3.3", [minority(2.5e9) / 1e22, ATOMS / minority(2.5e9) / 1e6])
assert agrees("P3.7", [0.15 / (2 * 60 * 0.5) * 1e6, 0.15 / (60 * 0.5) * 1e6])
v_low = peak(100) - 2 * DROP; i_low = 6.9 / v_low
assert agrees("P3.8", [v_low, i_low * 1e3, ripple(i_low, 50, 14.7e-6)])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## Three rectifiers, computed from the AC wave

A transformer's sine wave (ghosted), and what reaches a load resistor through a half-wave rectifier (one diode), a
centre-tap rectifier (two diodes, each half of the winding giving the full voltage) and a bridge (four diodes, two in
every path). At each instant the loop is solved with the diode equation, so the output's peak sits one diode drop, or
two, below the supply's. Change the supply and the load.

In [ ]:
def rect_waves(rms=12.0, R=1000.0, f=50.0, n=400):
    t = np.linspace(0, 2 / f, 2 * n + 1)
    v = peak(rms) * np.sin(2 * math.pi * f * t)
    half = np.array([solve_loop(x, R) * R if x > 0 else 0.0 for x in v])
    tap = np.array([solve_loop(abs(x), R, 1) * R for x in v])
    bridge = np.array([solve_loop(abs(x), R, 2) * R for x in v])
    return t, v, half, tap, bridge

def rect_picture(rms=12.0, R=1000.0):
    t, v, half, tap, bridge = rect_waves(rms, R)
    fig, axes = plt.subplots(3, 1, figsize=(10, 7.2), sharex=True)
    fig.subplots_adjust(left=0.1, right=0.97, top=0.90, bottom=0.09, hspace=0.2)
    for ax, y, name in zip(axes, (half, tap, bridge), ("half wave · 1 diode", "centre tap · 2 diodes", "bridge · 4 diodes")):
        ax.plot(t * 1e3, v, color=GHOST); ax.plot(t * 1e3, y, color=CHALK)
        ax.set_ylabel("volts"); ax.text(0.99, 0.08, f"{name} · peak {y.max():.2f} V · average {y[:-1].mean():.2f} V",
                                        transform=ax.transAxes, ha="right", color=GOLD, family="monospace", fontsize=11)
    axes[2].set_xlabel("time (ms)")
    fig.suptitle(f"supply {rms:.1f} V r m s, peak {peak(rms):.2f} V, 50 Hz · load {R:.0f} Ω", x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return half.max(), tap.max(), bridge.max(), half[:-1].mean(), bridge[:-1].mean()

h_pk, t_pk, b_pk, h_mean, b_mean = rect_picture(12.0, 1000.0)
assert round(h_pk, 1) == 16.2 and round(t_pk, 1) == 16.2 and round(b_pk, 1) == 15.5      # one drop, one drop, two drops
assert round(h_mean, 1) == 5.1 and round(b_mean, 1) == 9.4                               # the averages the course quotes
assert abs(h_mean / (peak(12) / math.pi) - 1) < 0.08 and abs(b_mean / (2 * peak(12) / math.pi) - 1) < 0.14   # near peak/π and 2 × peak/π

In [ ]:
interact(rect_picture, rms=FloatSlider(value=12.0, min=3.0, max=24.0, step=0.5, description="supply (V rms)"),
         R=FloatSlider(value=1000.0, min=100.0, max=5000.0, step=100.0, description="load (Ω)"));

## A smoothing capacitor, stepped through time

A bridge rectifier feeding a capacitor with a load across it. The capacitor's voltage is stepped forward a hundredth
of a millisecond at a time, with nothing but its charge balance: what the diodes pour in, minus what the load takes
out. Near each peak the diodes top it up in a short burst (the SKY spikes); between peaks it feeds the load alone, and
its voltage sinks: the **ripple**. The measured ripple is set beside the course's formula, ΔV = I/(2fC). Make the
capacitor small and watch the formula overshoot: it assumes the voltage barely sags.

In [ ]:
def filter_run(v_peak, f, C, R=None, P=None, drops=2, full=True, cycles=6, per=1000):
    """Backward Euler on C dV/dt = (the diodes' current) − (the load's current); the load is a resistor R, or draws a
    steady power P. Returns the last cycle: t, the rectified supply, the capacitor's voltage, the diodes' current."""
    w = 2 * math.pi * f; dt = 1 / f / per; V = 0.0; out = []
    for k in range(1, cycles * per + 1):
        t = k * dt; s = v_peak * math.sin(w * t)
        src = abs(s) if full else max(s, 0.0)
        def load(v_): return v_ / R if R is not None else (P / v_ if v_ > 1.0 else P * v_)
        def idio(v_): return I0 * (math.exp(min((src - v_) / (drops * VT), 700.0)) - 1.0)
        lo, hi = 0.0, max(V, src) + 1.0
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            if C * (mid - V) / dt - idio(mid) + load(mid) > 0: hi = mid
            else: lo = mid
            if hi - lo < 1e-10: break
        V = 0.5 * (lo + hi)
        if k > (cycles - 1) * per: out.append((t, src, V, max(idio(V), 0.0)))
    return np.array(out)

def filter_picture(C_uF=470.0, R=1000.0):
    run = filter_run(peak(12.0), 50.0, C_uF * 1e-6, R=R)
    t = (run[:, 0] - run[0, 0]) * 1e3
    measured = run[:, 2].max() - run[:, 2].min(); I = run[:, 2].mean() / R
    formula = ripple(I, 50.0, C_uF * 1e-6)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True)
    fig.subplots_adjust(left=0.1, right=0.97, top=0.88, bottom=0.1, hspace=0.16)
    a1.plot(t, run[:, 1], color=GHOST, label="the rectifier's humps"); a1.plot(t, run[:, 2], color=CHALK, label="the capacitor's voltage")
    a1.set_ylabel("volts"); a1.legend(frameon=False, loc="lower right")
    a2.plot(t, run[:, 3] * 1e3, color=SKY); a2.axhline(I * 1e3, color=DIM, linestyle="--")
    a2.set_ylabel("diodes' current (mA)"); a2.set_xlabel("time (ms), one cycle")
    fig.suptitle(f"{C_uF:.0f} µF, {R:.0f} Ω · load current {I * 1e3:.1f} mA · ripple {measured:.3f} V stepped, "
                 f"{formula:.3f} V by I/(2fC)", x=0.03, ha="left", color=GOLD)
    footnote(fig, "simulated · checked against the formula"); plt.show()
    return measured, formula, float((run[:, 3] > 1e-4).mean()), float(run[:, 3].max() / I)

measured, formula, on_share, burst = filter_picture(470.0, 1000.0)
assert round(measured, 2) == 0.29 and round(formula, 2) == 0.33         # the course's example: 0.29 V stepped, 0.33 V by the formula
assert on_share < 0.2 and round(burst) == 20                             # the diodes conduct under a fifth of the time, in bursts 20 times the load's current
small, small_formula, _, _ = filter_picture(10.0, 1000.0)
assert round(small) == 7 and round(small_formula) == 12                  # the edge: the formula fails when the ripple is large

In [ ]:
interact(filter_picture, C_uF=FloatSlider(value=470.0, min=10.0, max=2200.0, step=10.0, description="capacitor (µF)"),
         R=FloatSlider(value=1000.0, min=100.0, max=5000.0, step=100.0, description="load (Ω)"));

## The phone charger's first stage

The course's opening question, with a published design's values: a bridge of four diodes straight on the mains, then
14.7 µF of capacitance (4.7 µF and 10 µF side by side, rated 400 V), feeding a stage that draws a steady power (about
6.8 W on 230 V mains for 5 W out). Because that stage needs the same power whatever the voltage, it draws more
current when the mains is lower, and the ripple grows. Try 230 V at 50 Hz, then 120 V at 60 Hz, then 100 V at 50 Hz.
**Never open a real charger: mains voltage can kill.**

In [ ]:
def charger_picture(mains=230.0, hertz=50.0, C_uF=14.7, P_in=6.8):
    f = hertz
    run = filter_run(peak(mains), f, C_uF * 1e-6, P=P_in)
    t = (run[:, 0] - run[0, 0]) * 1e3
    v_pk = peak(mains) - 2 * DROP; I = P_in / v_pk; formula = ripple(I, f, C_uF * 1e-6)
    measured = run[:, 2].max() - run[:, 2].min()
    fig, ax = plt.subplots(figsize=(10, 4.4))
    fig.subplots_adjust(left=0.1, right=0.97, top=0.82, bottom=0.15)
    ax.plot(t, run[:, 1], color=GHOST, label="the rectified mains"); ax.plot(t, run[:, 2], color=CHALK, label="the capacitors' voltage")
    ax.set_xlabel("time (ms), one cycle"); ax.set_ylabel("volts"); ax.legend(frameon=False, loc="lower right")
    fig.suptitle(f"{mains:.0f} V r m s, {f:.0f} Hz · peak at the capacitors {v_pk:.0f} V · {I * 1e3:.0f} mA · ripple "
                 f"{formula:.1f} V by the formula, {measured:.1f} V stepped ({100 * formula / v_pk:.0f} % of the peak)",
                 x=0.03, ha="left", color=GOLD, fontsize=12)
    footnote(fig, "simulated · design values: Power Integrations RDR-158 · the fusible resistor and filter coils left out"); plt.show()
    return v_pk, I, formula, measured

v_pk, I_ch, dv, dv_sim = charger_picture(230.0, 50.0)
assert round(v_pk) == 324 and round(I_ch * 1e3) == 21 and round(dv) == 14 and round(dv_sim) == 13     # the course's answers
v_us, I_us, dv_us, _ = charger_picture(120.0, 60.0, P_in=6.9)
assert round(v_us) == 168 and round(I_us * 1e3) == 41 and round(dv_us) == 23
assert round(peak(265.0)) == 375 < 400                                                                # the capacitors' 400 V rating

In [ ]:
interact(charger_picture, mains=FloatSlider(value=230.0, min=85.0, max=265.0, step=5.0, description="mains (V rms)"),
         hertz=FloatSlider(value=50.0, min=50.0, max=60.0, step=10.0, description="frequency (Hz)"),
         C_uF=FloatSlider(value=14.7, min=4.7, max=47.0, step=0.5, description="capacitance (µF)"),
         P_in=FloatSlider(value=6.8, min=1.0, max=12.0, step=0.1, description="power drawn (W)"));

## Sources
Every exercise and set-up here is our own. Silicon's and germanium's band gaps (1.12 eV and 0.66 eV at 300 K), atom
densities, intrinsic carrier densities (1 × 10¹⁶ and 2 × 10¹⁹ per m³), donor and acceptor energies and silicon's
permittivity (11.7) are from the Ioffe Institute's NSM archive (ioffe.ru/SVA/NSM/Semicond); other tables give pure
silicon between 0.97 and 1.5 × 10¹⁶ per m³. The thermal voltage kT/e = 25.85 mV at 300 K comes from the CODATA 2022
constants. The diode equation is William Shockley's (Bell System Technical Journal 28, 1949); the course's diode,
with I₀ = 10⁻¹⁴ A, is an ideal one, and real diodes need a little more than 60 mV for a tenfold step. The junction's
barrier and width formulas are the standard ones (for example pveducation.org, "Solving for depletion region"). The
"about 0.7 V for silicon, 0.2 V for germanium" threshold voltages are typical textbook values. The charger's values
(four 1N4007 diodes, 4.7 µF and 10 µF at 400 V, 85–265 V AC, 73.5 % efficient at 230 V and full load) are from Power
Integrations' reference design report RDR-158 (2008); its 10 Ω fusible resistor and its two filter coils are left out
of the sums here. Mains voltages are the nominal values of IEC 60038. The solvers are the course's: Newton's method on
Poisson's equation with the Boltzmann factor; halving for a diode in a loop; backward Euler for the capacitor.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph23_semiconductors/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph23_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert (VT, I0, NI, ATOMS, DROP) == (_P.VT, _P.I0, _P.NI["Si"], _P.ATOMS["Si"], _P.V_DROP)
    assert abs(V0 - _P.SIM["barrier"]) < 1e-6 and abs(measured - _P.FILTER["ripple"]) < 1e-6
    print("This notebook matches the course's practice set and solvers.")